In [ ]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/D-alanine_295K_278464_invert_cb_rot_17O_opt_magres_new.magres') #latest magres file from 2025

In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-58.71056973 168.42333854  80.19168888]
 [175.6277732   48.84724812 -29.02649162]
 [ -9.58999475  25.6436215  -52.86906924]]

17O2 sigma:
 [[ -58.71056973 -168.42333854  -80.19168888]
 [-175.6277732    48.84724812  -29.02649162]
 [   9.58999475   25.6436215   -52.86906924]]

17O3 sigma:
 [[-58.71056973 168.42333854 -80.19168888]
 [175.6277732   48.84724812  29.02649162]
 [  9.58999475 -25.6436215  -52.86906924]]

17O4 sigma:
 [[ -58.71056973 -168.42333854   80.19168888]
 [-175.6277732    48.84724812   29.02649162]
 [  -9.58999475  -25.6436215   -52.86906924]]

17O5 sigma:
 [[ -47.51972884  228.54720094  -55.0666635 ]
 [ 197.59879891  101.82413744   88.59783075]
 [ -16.91499843   50.38859125 -180.28276326]]

17O6 sigma:
 [[ -47.51972884 -228.54720094   55.0666635 ]
 [-197.59879891  101.82413744   88.59783075]
 [  16.91499843   50.38859125 -180.28276326]]

17O7 sigma:
 [[ -47.51972884  228.54720094   55.0666635 ]
 [ 197.59879891  101.82413744  -88.59783075]
 [  16.91499843

In [7]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.353653917778783

17O2 sigma:
 6.353653917778768

17O3 sigma:
 6.353653917778764

17O4 sigma:
 6.353653917778741

17O5 sigma:
 8.336943250025397

17O6 sigma:
 8.336943250025401

17O7 sigma:
 8.336943250025469

17O8 sigma:
 8.336943250025486



In [8]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

atom_label = 4
CS_total[:,:] = atoms.species('O').ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('O')[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = -0.0256 #electric quadrupole moment for O17 in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[-4.222  0.941  0.784]
 [ 0.941 -3.888 -1.556]
 [ 0.784 -1.556  8.11 ]]

CS Tensor:
 [[ -47.52   228.547  -55.067]
 [ 197.599  101.824   88.598]
 [ -16.915   50.389 -180.283]]

CS isotropic Tensor:
 [[-41.993   0.      0.   ]
 [  0.    -41.993   0.   ]
 [  0.      0.    -41.993]]

CS symmetric Tensor:
 [[ -47.52   213.073  -35.991]
 [ 213.073  101.824   69.493]
 [ -35.991   69.493 -180.283]]

CS antisymmetric Tensor:
 [[  0.     15.474 -19.076]
 [-15.474   0.     19.105]
 [ 19.076 -19.105   0.   ]]


In [9]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [-5.20600843 -3.13745029  8.34345872] 

 Unsorted Eigenvectors:
 [[-0.73472195  0.6763169   0.05271734]
 [ 0.66744329  0.73459395 -0.12202941]
 [ 0.12125639  0.05447185  0.99112547]] 

Sorted Eigenvalues: 
 [-3.13745029 -5.20600843  8.34345872] 

Sorted Eigenvectors: 
 [[ 0.6763169  -0.73472195  0.05271734]
 [ 0.73459395  0.66744329 -0.12202941]
 [ 0.05447185  0.12125639  0.99112547]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 255.95335074 -121.21765582 -260.71404959] 

 Unsorted Eigenvectors:
 [[-0.56590489 -0.58495889  0.58101184]
 [-0.82018355  0.327649   -0.46898302]
 [-0.08396783  0.74193614  0.66519183]] 

Sorted Eigenvalues: 
 [-121.21765582 -260.71404959  255.95335074] 

Sorted Eigenvectors: 
 [[-0.58495889  0.58101184 -0.56590489]
 [ 0.327649   -0.46898302 -0.82018355]
 [ 0.74193614  0.66519183 -0.08396783]] 



In [10]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -3.137450293278898 -5.206008427497975 8.343458720776908
CSA Tensor Components δyy, δxx, δzz: 
 -121.21765582086562 -260.7140495941153 255.95335074403212


In [11]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        8.34346
etaq            0.247926
iso_cs (ppm)  -41.9928
csa (ppm)     297.946
etas            0.468193


In [12]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.73472195  0.6763169   0.05271734]
 [ 0.66744329  0.73459395 -0.12202941]
 [ 0.12125639  0.05447185  0.99112547]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
24.190984242353533 7.63891923516269 66.6354043334518 

Direction cosine csa: 

[[ 0.58101184 -0.58495889 -0.56590489]
 [-0.46898302  0.327649   -0.82018355]
 [ 0.66519183  0.74193614 -0.08396783]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
48.12180045610134 94.81667394094437 -55.395354608119085 



In [13]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 41.65068023434515 chi: 90.74309807148981 xi: -81.84584972526785 



**Rotation of tensors Crystal--> Tenon Frame**

In [14]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[ -47.51972884  213.07299992  -35.99083097]
 [ 213.07299992  101.82413744   69.493211  ]
 [ -35.99083097   69.493211   -180.28276326]]
CSA Tensor in Tenon Frame: 
 [[ 131.1284975   -64.66704208 -208.2960343 ]
 [ -64.66704208 -107.21477515   30.67734829]
 [-208.2960343    30.67734829 -149.89207702]]
Quad Tensor in Crystal Frame: 
 [[-4.22218497  0.94053292  0.78415903]
 [ 0.94053292 -3.88798846 -1.55598733]
 [ 0.78415903 -1.55598733  8.11017343]]
Quad Tensor in Tenon Frame: 
 [[-3.13200761 -2.52207408  1.49932622]
 [-2.52207408  0.40385435 -5.99693079]
 [ 1.49932622 -5.99693079  2.72815326]]
